In [ ]:
import sqlite3

# add file path to lahman_1871-2022.sqlite
conn = sqlite3.connect("<file path to lahman_1871-2022.sqlite>")
cursor = conn.cursor()

In [12]:
cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type='table';
""")

print(cursor.fetchall())

[('AllstarFull',), ('Appearances',), ('AwardsManagers',), ('AwardsPlayers',), ('AwardsShareManagers',), ('AwardsSharePlayers',), ('Batting',), ('BattingPost',), ('CollegePlaying',), ('Fielding',), ('FieldingOF',), ('FieldingOFsplit',), ('FieldingPost',), ('HallOfFame',), ('HomeGames',), ('Managers',), ('ManagersHalf',), ('Parks',), ('People',), ('Pitching',), ('PitchingPost',), ('Salaries',), ('Schools',), ('SeriesPost',), ('Teams',), ('TeamsFranchises',), ('TeamsHalf',)]


# Predict Player Salary from Batting Statistics

In [17]:
import pandas as pd

query = """
SELECT
    yearID,
    teamID,
    R,
    H,
    HR,
    BB,
    SO,
    ERA,
    W
FROM Teams
WHERE yearID > 1990
"""

df = pd.read_sql(query, conn)
df

,yearID,teamID,R,H,HR,BB,SO,ERA,W
0,1991,BAL,686,1421,170,528,974,4.59,67
1,1991,BOS,731,1486,126,593,820,4.01,84
2,1991,CAL,653,1396,115,448,928,3.69,81
3,1991,CHA,758,1464,139,610,896,3.79,87
4,1991,CLE,576,1390,79,449,888,4.23,57
...,...,...,...,...,...,...,...,...,...
937,2022,PIT,591,1186,158,476,1497,4.66,62
938,2022,SDN,705,1317,153,574,1327,3.81,89
939,2022,SFN,716,1261,183,571,1462,3.85,81
940,2022,SLN,772,1386,197,537,1226,3.79,93


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

X = df[["R","H","HR","BB","SO","ERA"]]
y = df["W"]

X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

model = RandomForestRegressor(n_estimators=200)
model.fit(X_train,y_train)

preds = model.predict(X_test)

print("R2:", r2_score(y_test,preds))

R2: 0.8947174027972167


In [19]:
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance)

  feature  importance
0       R    0.347500
5     ERA    0.288227
1       H    0.260322
3      BB    0.074343
4      SO    0.016521
2      HR    0.013086
